# PyTorch TCN 时序预测：从因果卷积到流式窗口

Temporal Convolutional Network 不是普通 Conv1d 堆叠。时序预测要求位置 t 的表示不能读取 t+1 之后的信息，因此 padding、窗口、标准化和验证切分都是架构合同。

本 Notebook 用基础 PyTorch 算子实现 CausalConv1d、残差 TemporalBlock 和 TCNForecaster，并验证因果性、感受野、梯度、严格时间评估、流式幂等与制品指纹。数据高度规则化，只用于验证实现，不能代表真实业务泛化。

## 1. 可复现环境

默认 CPU、固定随机种子和单线程，不把 CUDA 当隐式依赖。真实 GPU 训练还要绑定 CUDA、cuDNN、精度模式、硬件与 deterministic 配置。

In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message="The pynvml package is deprecated", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

from dataclasses import dataclass, field  # 导入本单元所需的依赖。
from hashlib import sha256  # 导入本单元所需的依赖。
from copy import deepcopy  # 导入本单元所需的依赖。
from datetime import datetime, timedelta, timezone  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 2901  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
np.random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。
assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.__version__  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "seed": SEED, "device": str(DEVICE)})  # 执行当前语句以推进本节示例。

## 2. 数据与标签时点

生成 30 天 hourly gauge：日周期、周周期、趋势和噪声。每个样本用过去 32 点预测下一点。预测目标位置小于 480 属于 train，480 到 599 属于 validation，600 之后属于 test。所有可学习统计量只在 train 拟合。

In [ ]:
N = 24 * 30  # 计算并保存当前步骤的中间状态。
time_index = np.arange(N)  # 计算并保存当前步骤的中间状态。
hour = time_index % 24  # 计算并保存当前步骤的中间状态。
day = time_index / 24  # 计算并保存当前步骤的中间状态。
rng = np.random.default_rng(SEED)  # 计算并保存当前步骤的中间状态。
raw_value = (  # 计算并保存当前步骤的中间状态。
    50 + 8 * np.sin(2 * np.pi * hour / 24)  # 执行当前语句以推进本节示例。
    + 2.5 * np.sin(2 * np.pi * day / 7)  # 执行当前语句以推进本节示例。
    + 0.015 * time_index + rng.normal(0, 0.35, N)  # 执行当前语句以推进本节示例。
).astype(np.float32)  # 执行当前语句以推进本节示例。
timestamps = [  # 计算并保存当前步骤的中间状态。
    datetime(2026, 1, 1, tzinfo=timezone.utc) + timedelta(hours=int(i))  # 计算并保存当前步骤的中间状态。
    for i in time_index  # 遍历输入元素以累积或检查结果。
]  # 执行当前语句以推进本节示例。
TRAIN_END, VALID_END, WINDOW = 480, 600, 32  # 计算并保存当前步骤的中间状态。
train_mean = float(raw_value[:TRAIN_END].mean())  # 计算并保存当前步骤的中间状态。
train_std = float(raw_value[:TRAIN_END].std())  # 计算并保存当前步骤的中间状态。
normalized_value = (raw_value - train_mean) / train_std  # 计算并保存当前步骤的中间状态。
known_time_features = np.stack([  # 计算并保存当前步骤的中间状态。
    np.sin(2 * np.pi * hour / 24),  # 执行当前语句以推进本节示例。
    np.cos(2 * np.pi * hour / 24),  # 执行当前语句以推进本节示例。
], axis=1).astype(np.float32)  # 计算并保存当前步骤的中间状态。
all_features = np.concatenate([normalized_value[:, None], known_time_features], axis=1)  # 计算并保存当前步骤的中间状态。
assert TRAIN_END < VALID_END < N  # 用受控断言验证关键不变量。
assert timestamps[TRAIN_END - 1] < timestamps[TRAIN_END] < timestamps[VALID_END]  # 用受控断言验证关键不变量。
assert train_std > 0  # 用受控断言验证关键不变量。
assert all_features.shape == (N, 3)  # 用受控断言验证关键不变量。
print({"points": N, "train_mean": train_mean, "train_std": train_std})  # 执行当前语句以推进本节示例。

## 3. 因果监督窗口

输入 shape 约定为 batch、channels、length。target 是窗口右边界的下一点。保留 target index 用于审计。随机切分高度重叠窗口会造成泄漏，因此这里只按 target time 切分。

In [ ]:
def make_windows(features, targets, window):  # 定义本节可复用的核心函数。
    xs, ys, indices = [], [], []  # 计算并保存当前步骤的中间状态。
    for target_index in range(window, len(targets)):  # 遍历输入元素以累积或检查结果。
        xs.append(features[target_index-window:target_index].T)  # 执行当前语句以推进本节示例。
        ys.append(targets[target_index])  # 执行当前语句以推进本节示例。
        indices.append(target_index)  # 执行当前语句以推进本节示例。
    return (  # 返回当前分支计算出的结果。
        torch.tensor(np.stack(xs), dtype=torch.float32),  # 计算并保存当前步骤的中间状态。
        torch.tensor(ys, dtype=torch.float32),  # 计算并保存当前步骤的中间状态。
        np.asarray(indices),  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。

X_all, y_all, target_indices = make_windows(all_features, normalized_value, WINDOW)  # 计算并保存当前步骤的中间状态。
train_mask = target_indices < TRAIN_END  # 计算并保存当前步骤的中间状态。
valid_mask = (target_indices >= TRAIN_END) & (target_indices < VALID_END)  # 计算并保存当前步骤的中间状态。
test_mask = target_indices >= VALID_END  # 计算并保存当前步骤的中间状态。
X_train, y_train = X_all[train_mask], y_all[train_mask]  # 计算并保存当前步骤的中间状态。
X_valid, y_valid = X_all[valid_mask], y_all[valid_mask]  # 计算并保存当前步骤的中间状态。
X_test, y_test = X_all[test_mask], y_all[test_mask]  # 计算并保存当前步骤的中间状态。
assert X_all.shape[1:] == (3, WINDOW)  # 用受控断言验证关键不变量。
assert len(X_train) + len(X_valid) + len(X_test) == len(X_all)  # 用受控断言验证关键不变量。
assert target_indices[train_mask].max() < target_indices[valid_mask].min()  # 用受控断言验证关键不变量。
assert target_indices[valid_mask].max() < target_indices[test_mask].min()  # 用受控断言验证关键不变量。
assert torch.allclose(X_all[0, 0], torch.tensor(normalized_value[:WINDOW]))  # 用受控断言验证关键不变量。
print({"train": len(X_train), "validation": len(X_valid), "test": len(X_test)})  # 执行当前语句以推进本节示例。

## 4. CausalConv1d

kernel size k、dilation d 的单层只在左侧补 (k-1)d 个零。先调用 F.pad，再使用 padding 为零的 Conv1d，使输出长度等于输入且当前位置不读取未来。

In [ ]:
class CausalConv1d(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1, bias=True):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if kernel_size < 1 or dilation < 1:  # 按当前条件选择后续控制路径。
            raise ValueError("kernel_and_dilation_must_be_positive")  # 遇到非法合同立即显式失败。
        self.left_padding = (kernel_size - 1) * dilation  # 计算并保存当前步骤的中间状态。
        self.conv = nn.Conv1d(  # 计算并保存当前步骤的中间状态。
            in_channels, out_channels, kernel_size,  # 执行当前语句以推进本节示例。
            padding=0, dilation=dilation, bias=bias,  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。

    def forward(self, x):  # 定义本节可复用的核心函数。
        if x.ndim != 3:  # 按当前条件选择后续控制路径。
            raise ValueError("expected_BCL")  # 遇到非法合同立即显式失败。
        return self.conv(F.pad(x, (self.left_padding, 0)))  # 返回当前分支计算出的结果。

causal_probe = CausalConv1d(2, 4, kernel_size=3, dilation=2)  # 计算并保存当前步骤的中间状态。
assert causal_probe(torch.randn(5, 2, 17)).shape == (5, 4, 17)  # 用受控断言验证关键不变量。
assert causal_probe.left_padding == 4  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    CausalConv1d(1, 1, 0)  # 执行当前语句以推进本节示例。
    raise AssertionError("zero kernel must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

## 5. 残差 TemporalBlock 与感受野

每个 block 含两次相同 dilation 的因果卷积。channel 不同时用 1x1 卷积投影残差。三层 dilation 为 1、2、4，kernel 为 3 时，感受野 R = 1 + 2 x (3-1) x (1+2+4) = 29。

In [ ]:
class TemporalBlock(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.0):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.conv1 = CausalConv1d(in_channels, out_channels, kernel_size, dilation)  # 计算并保存当前步骤的中间状态。
        self.conv2 = CausalConv1d(out_channels, out_channels, kernel_size, dilation)  # 计算并保存当前步骤的中间状态。
        self.dropout = nn.Dropout(dropout)  # 计算并保存当前步骤的中间状态。
        self.activation = nn.ReLU()  # 计算并保存当前步骤的中间状态。
        self.residual = (  # 计算并保存当前步骤的中间状态。
            nn.Identity() if in_channels == out_channels  # 计算并保存当前步骤的中间状态。
            else nn.Conv1d(in_channels, out_channels, kernel_size=1)  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。

    def forward(self, x):  # 定义本节可复用的核心函数。
        residual = self.residual(x)  # 计算并保存当前步骤的中间状态。
        hidden = self.dropout(self.activation(self.conv1(x)))  # 计算并保存当前步骤的中间状态。
        hidden = self.dropout(self.activation(self.conv2(hidden)))  # 计算并保存当前步骤的中间状态。
        if hidden.shape != residual.shape:  # 按当前条件选择后续控制路径。
            raise RuntimeError("residual_shape_mismatch")  # 遇到非法合同立即显式失败。
        return self.activation(hidden + residual)  # 返回当前分支计算出的结果。

def receptive_field(kernel_size, dilations, convolutions_per_block=2):  # 定义本节可复用的核心函数。
    return 1 + convolutions_per_block * (kernel_size - 1) * sum(dilations)  # 返回当前分支计算出的结果。

block = TemporalBlock(3, 8, kernel_size=3, dilation=2)  # 计算并保存当前步骤的中间状态。
assert block(torch.randn(2, 3, 19)).shape == (2, 8, 19)  # 用受控断言验证关键不变量。
assert receptive_field(3, [1, 2, 4]) == 29  # 用受控断言验证关键不变量。

## 6. TCNForecaster.forward

模型保留完整 sequence 输出，最后取最右位置预测下一点。encode_sequence 用于直接做因果干预测试。网络输出保持在标准化空间，服务 bundle 负责反标准化。

In [ ]:
class TCNForecaster(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, input_channels, channels=(16, 16, 16), kernel_size=3, dropout=0.0):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.input_channels = int(input_channels)  # 计算并保存当前步骤的中间状态。
        self.channels = tuple(int(channel) for channel in channels)  # 计算并保存当前步骤的中间状态。
        self.kernel_size = int(kernel_size)  # 计算并保存当前步骤的中间状态。
        self.dropout = float(dropout)  # 计算并保存当前步骤的中间状态。
        blocks, in_channels, dilations = [], self.input_channels, []  # 计算并保存当前步骤的中间状态。
        for level, out_channels in enumerate(self.channels):  # 遍历输入元素以累积或检查结果。
            dilation = 2 ** level  # 计算并保存当前步骤的中间状态。
            blocks.append(TemporalBlock(in_channels, out_channels, self.kernel_size, dilation, self.dropout))  # 执行当前语句以推进本节示例。
            dilations.append(dilation)  # 执行当前语句以推进本节示例。
            in_channels = out_channels  # 计算并保存当前步骤的中间状态。
        self.network = nn.Sequential(*blocks)  # 计算并保存当前步骤的中间状态。
        self.head = nn.Conv1d(in_channels, 1, kernel_size=1)  # 计算并保存当前步骤的中间状态。
        self.dilations = tuple(dilations)  # 计算并保存当前步骤的中间状态。

    def encode_sequence(self, x):  # 定义本节可复用的核心函数。
        return self.head(self.network(x))  # 返回当前分支计算出的结果。

    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.encode_sequence(x)[:, 0, -1]  # 返回当前分支计算出的结果。

model = TCNForecaster(3).to(DEVICE)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    model_probe = model(torch.randn(7, 3, WINDOW))  # 计算并保存当前步骤的中间状态。
parameter_count = sum(p.numel() for p in model.parameters())  # 计算并保存当前步骤的中间状态。
assert model_probe.shape == (7,)  # 用受控断言验证关键不变量。
assert receptive_field(model.kernel_size, model.dilations) == 29  # 用受控断言验证关键不变量。
assert model.channels == (16, 16, 16) and model.dropout == 0.0  # 用受控断言验证关键不变量。
assert parameter_count > 1000  # 用受控断言验证关键不变量。
print({"parameters": parameter_count, "receptive_field": 29})  # 执行当前语句以推进本节示例。

## 7. 行为级因果测试

复制输入后只修改位置 20 及之后。严格因果模型在位置 0 到 19 的输出必须相同。再用对称 padding 卷积构造失败反例：它在位置 19 会读到位置 20。

In [ ]:
model.eval()  # 执行当前语句以推进本节示例。
causal_input = torch.randn(1, 3, 32)  # 计算并保存当前步骤的中间状态。
changed_future = causal_input.clone()  # 计算并保存当前步骤的中间状态。
changed_future[:, :, 20:] += 100  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    causal_left = model.encode_sequence(causal_input)  # 计算并保存当前步骤的中间状态。
    causal_right = model.encode_sequence(changed_future)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(causal_left[:, :, :20], causal_right[:, :, :20], atol=1e-6)  # 用受控断言验证关键不变量。

bad_conv = nn.Conv1d(1, 1, kernel_size=3, padding=1, bias=False)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    bad_conv.weight.fill_(1.0)  # 执行当前语句以推进本节示例。
bad_input = torch.zeros(1, 1, 25)  # 计算并保存当前步骤的中间状态。
bad_changed = bad_input.clone()  # 计算并保存当前步骤的中间状态。
bad_changed[:, :, 20] = 5  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    bad_before = bad_conv(bad_input)  # 计算并保存当前步骤的中间状态。
    bad_after = bad_conv(bad_changed)  # 计算并保存当前步骤的中间状态。
assert not torch.allclose(bad_before[:, :, 19], bad_after[:, :, 19])  # 用受控断言验证关键不变量。
assert torch.allclose(causal_left[:, :, 19], causal_right[:, :, 19], atol=1e-6)  # 用受控断言验证关键不变量。

## 8. 训练与 validation checkpoint

只有 train window 参与反向传播。每 10 步在 validation 计算 MSE 并保存最佳 state_dict。test 在模型与决策冻结后只评一次。本例使用 full batch 以缩短演示时间。

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # 计算并保存当前步骤的中间状态。
loss_fn = nn.MSELoss()  # 计算并保存当前步骤的中间状态。
train_losses, validation_history = [], []  # 计算并保存当前步骤的中间状态。
best_validation, best_state = float("inf"), None  # 计算并保存当前步骤的中间状态。

for step in range(181):  # 遍历输入元素以累积或检查结果。
    model.train()  # 执行当前语句以推进本节示例。
    optimizer.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    loss = loss_fn(model(X_train), y_train)  # 计算并保存当前步骤的中间状态。
    loss.backward()  # 执行当前语句以推进本节示例。
    if step == 0:  # 按当前条件选择后续控制路径。
        first_grad_norm = torch.sqrt(sum(  # 计算并保存当前步骤的中间状态。
            (p.grad.detach() ** 2).sum() for p in model.parameters() if p.grad is not None  # 执行当前语句以推进本节示例。
        ))  # 执行当前语句以推进本节示例。
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)  # 执行当前语句以推进本节示例。
    optimizer.step()  # 执行当前语句以推进本节示例。
    train_losses.append(float(loss.detach()))  # 执行当前语句以推进本节示例。
    if step % 10 == 0:  # 按当前条件选择后续控制路径。
        model.eval()  # 执行当前语句以推进本节示例。
        with torch.no_grad():  # 在受管理的上下文中执行操作。
            validation_loss = float(loss_fn(model(X_valid), y_valid))  # 计算并保存当前步骤的中间状态。
        validation_history.append((step, validation_loss))  # 执行当前语句以推进本节示例。
        if validation_loss < best_validation:  # 按当前条件选择后续控制路径。
            best_validation = validation_loss  # 计算并保存当前步骤的中间状态。
            best_state = deepcopy(model.state_dict())  # 计算并保存当前步骤的中间状态。

assert best_state is not None  # 用受控断言验证关键不变量。
model.load_state_dict(best_state)  # 执行当前语句以推进本节示例。
assert train_losses[-1] < train_losses[0] * 0.15  # 用受控断言验证关键不变量。
assert first_grad_norm > 0 and torch.isfinite(first_grad_norm)  # 用受控断言验证关键不变量。
print({"train_mse": [round(train_losses[0], 4), round(train_losses[-1], 4)],  # 执行当前语句以推进本节示例。
       "best_validation_mse": round(best_validation, 4)})  # 执行当前语句以推进本节示例。

## 9. 冻结 test 与季节 baseline

将预测反标准化回原单位，报告 MAE 与 RMSE。这里的 test 是 **rolling one-step teacher forcing**：预测每个时点时，窗口可使用此前已经真实到达的观测；它不是把前一步预测递归塞回窗口的 recursive multi-horizon。baseline 使用 24 小时前真实值，它在预测时已可见。若业务要求一次性预测未来 24 点，必须另建 recursive/direct multi-horizon 评估，不能直接复用这里的数字。生产还需要 rolling-origin、分桶和置信区间。

In [ ]:
model.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    test_prediction_z = model(X_test).cpu().numpy()  # 计算并保存当前步骤的中间状态。
test_prediction = test_prediction_z * train_std + train_mean  # 计算并保存当前步骤的中间状态。
test_target = y_test.numpy() * train_std + train_mean  # 计算并保存当前步骤的中间状态。
test_target_indices = target_indices[test_mask]  # 计算并保存当前步骤的中间状态。
seasonal_baseline = raw_value[test_target_indices - 24]  # 计算并保存当前步骤的中间状态。

def regression_metrics(target, prediction):  # 定义本节可复用的核心函数。
    error = prediction - target  # 计算并保存当前步骤的中间状态。
    return {  # 返回当前分支计算出的结果。
        "mae": float(np.mean(np.abs(error))),  # 执行当前语句以推进本节示例。
        "rmse": float(np.sqrt(np.mean(error ** 2))),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。

tcn_metrics = regression_metrics(test_target, test_prediction)  # 计算并保存当前步骤的中间状态。
baseline_metrics = regression_metrics(test_target, seasonal_baseline)  # 计算并保存当前步骤的中间状态。
print({"TCN": tcn_metrics, "seasonal_24h": baseline_metrics})  # 执行当前语句以推进本节示例。
assert all(math.isfinite(v) and v >= 0 for v in tcn_metrics.values())  # 用受控断言验证关键不变量。
assert tcn_metrics["mae"] < 1.5  # 用受控断言验证关键不变量。
assert len(test_prediction) == len(X_test)  # 用受控断言验证关键不变量。

## 10. 发布制品、数据快照与受信注册表

可追踪哈希只有在推理入口真正校验时才有意义。本节把模型完整配置、训练/切分配置、feature 公式与顺序、数据和 feature 快照、权重 dtype/shape/bytes、各层 version 一起写入 bundle；受信注册表保存发布时的摘要与租户/序列授权。validate_artifact 每次推理前都重新计算 bundle、配置和当前模型权重，并与受信条目比对，未知制品、参数篡改、数据快照篡改和自洽但未注册的伪造 bundle 都 fail-closed。示例注册表只是内存版控制面；生产中应由只读权限、签名/KMS 和审计日志保护。

In [ ]:
def canonical_hash(payload):  # 定义本节可复用的核心函数。
    encoded = json.dumps(  # 计算并保存当前步骤的中间状态。
        payload, sort_keys=True, ensure_ascii=False,  # 计算并保存当前步骤的中间状态。
        separators=(",", ":"), allow_nan=False,  # 计算并保存当前步骤的中间状态。
    ).encode("utf-8")  # 执行当前语句以推进本节示例。
    return sha256(encoded).hexdigest()  # 返回当前分支计算出的结果。


def array_hash(array):  # 定义本节可复用的核心函数。
    contiguous = np.ascontiguousarray(array)  # 计算并保存当前步骤的中间状态。
    metadata = {  # 计算并保存当前步骤的中间状态。
        "dtype": str(contiguous.dtype),  # 执行当前语句以推进本节示例。
        "shape": list(contiguous.shape),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    digest = sha256(json.dumps(metadata, sort_keys=True).encode("utf-8"))  # 计算并保存当前步骤的中间状态。
    digest.update(contiguous.tobytes())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。


def state_dict_hash(module):  # 定义本节可复用的核心函数。
    digest = sha256()  # 计算并保存当前步骤的中间状态。
    for name, tensor in sorted(module.state_dict().items()):  # 遍历输入元素以累积或检查结果。
        value = tensor.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        metadata = {  # 计算并保存当前步骤的中间状态。
            "name": name,  # 执行当前语句以推进本节示例。
            "dtype": str(value.dtype),  # 执行当前语句以推进本节示例。
            "shape": list(value.shape),  # 执行当前语句以推进本节示例。
        }  # 执行当前语句以推进本节示例。
        digest.update(json.dumps(metadata, sort_keys=True).encode("utf-8"))  # 计算并保存当前步骤的中间状态。
        digest.update(value.numpy().tobytes())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。


def model_config(module):  # 定义本节可复用的核心函数。
    if type(module) is not TCNForecaster:  # 按当前条件选择后续控制路径。
        raise TypeError("untrusted_model_class")  # 遇到非法合同立即显式失败。
    return {  # 返回当前分支计算出的结果。
        "architecture": "TCNForecaster",  # 执行当前语句以推进本节示例。
        "input_channels": module.input_channels,  # 执行当前语句以推进本节示例。
        "channels": list(module.channels),  # 执行当前语句以推进本节示例。
        "kernel_size": module.kernel_size,  # 执行当前语句以推进本节示例。
        "dilations": list(module.dilations),  # 执行当前语句以推进本节示例。
        "dropout": module.dropout,  # 执行当前语句以推进本节示例。
        "receptive_field": receptive_field(module.kernel_size, module.dilations),  # 执行当前语句以推进本节示例。
        "output": "next_value_z",  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。


def make_training_snapshot(raw, features, stamp_sequence, supervised_indices):  # 定义本节可复用的核心函数。
    timestamp_payload = [stamp.astimezone(timezone.utc).isoformat() for stamp in stamp_sequence]  # 计算并保存当前步骤的中间状态。
    body = {  # 计算并保存当前步骤的中间状态。
        "raw_value": {"sha256": array_hash(raw), "count": int(len(raw))},  # 执行当前语句以推进本节示例。
        "feature_matrix": {  # 执行当前语句以推进本节示例。
            "sha256": array_hash(features),  # 执行当前语句以推进本节示例。
            "shape": list(features.shape),  # 执行当前语句以推进本节示例。
        },  # 执行当前语句以推进本节示例。
        "timestamps_sha256": canonical_hash(timestamp_payload),  # 执行当前语句以推进本节示例。
        "target_indices_sha256": array_hash(np.asarray(supervised_indices, dtype=np.int64)),  # 计算并保存当前步骤的中间状态。
    }  # 执行当前语句以推进本节示例。
    return {**body, "snapshot_sha256": canonical_hash(body)}  # 返回当前分支计算出的结果。


MODEL_AND_PIPELINE_CONFIG = {  # 计算并保存当前步骤的中间状态。
    "model": model_config(model),  # 执行当前语句以推进本节示例。
    "features": {  # 执行当前语句以推进本节示例。
        "feature_version": "hour-cyclic-utc-v1",  # 执行当前语句以推进本节示例。
        "order": ["value_z", "hour_sin", "hour_cos"],  # 执行当前语句以推进本节示例。
        "timezone": "UTC",  # 执行当前语句以推进本节示例。
        "hour_period": 24,  # 执行当前语句以推进本节示例。
        "value_normalization": "train-only-zscore",  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
    "sampling": {  # 执行当前语句以推进本节示例。
        "window": WINDOW,  # 执行当前语句以推进本节示例。
        "frequency_label": "1h",  # 执行当前语句以推进本节示例。
        "frequency_seconds": 3600,  # 执行当前语句以推进本节示例。
        "target_offset_steps": 1,  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
    "split": {  # 执行当前语句以推进本节示例。
        "train_target_end_exclusive": TRAIN_END,  # 执行当前语句以推进本节示例。
        "validation_target_end_exclusive": VALID_END,  # 执行当前语句以推进本节示例。
        "test_target_start_inclusive": VALID_END,  # 执行当前语句以推进本节示例。
        "policy": "ordered-by-target-time",  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
    "training": {  # 执行当前语句以推进本节示例。
        "seed": SEED,  # 执行当前语句以推进本节示例。
        "optimizer": "Adam",  # 执行当前语句以推进本节示例。
        "learning_rate": 0.01,  # 执行当前语句以推进本节示例。
        "steps": 181,  # 执行当前语句以推进本节示例。
        "loss": "MSE",  # 执行当前语句以推进本节示例。
        "gradient_clip_norm": 5.0,  # 执行当前语句以推进本节示例。
        "validation_interval_steps": 10,  # 执行当前语句以推进本节示例。
        "checkpoint_policy": "minimum-validation-MSE",  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
    "runtime": {  # 执行当前语句以推进本节示例。
        "framework": "PyTorch",  # 执行当前语句以推进本节示例。
        "torch_version": torch.__version__,  # 执行当前语句以推进本节示例。
        "dtype": "float32",  # 执行当前语句以推进本节示例。
        "device_contract": "cpu",  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
training_snapshot = make_training_snapshot(  # 计算并保存当前步骤的中间状态。
    raw_value, all_features, timestamps, target_indices,  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
artifact_core = {  # 计算并保存当前步骤的中间状态。
    "artifact_id": "tcn-hourly-v1@bundle-1",  # 执行当前语句以推进本节示例。
    "versions": {  # 执行当前语句以推进本节示例。
        "artifact_schema": "tcn-artifact.v1",  # 执行当前语句以推进本节示例。
        "model": "tcn-hourly-v1",  # 执行当前语句以推进本节示例。
        "data": "synthetic-hourly-gauge-v1",  # 执行当前语句以推进本节示例。
        "features": "hour-cyclic-utc-v1",  # 执行当前语句以推进本节示例。
        "serving_contract": "strict-stream-one-step.v1",  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
    "configuration": MODEL_AND_PIPELINE_CONFIG,  # 执行当前语句以推进本节示例。
    "config_sha256": canonical_hash(MODEL_AND_PIPELINE_CONFIG),  # 执行当前语句以推进本节示例。
    "normalizer": {"mean": train_mean, "std": train_std},  # 执行当前语句以推进本节示例。
    "train_end_utc": timestamps[TRAIN_END - 1].isoformat(),  # 执行当前语句以推进本节示例。
    "training_snapshot": training_snapshot,  # 执行当前语句以推进本节示例。
    "state_dict_sha256": state_dict_hash(model),  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
artifact = {**artifact_core, "bundle_sha256": canonical_hash(artifact_core)}  # 计算并保存当前步骤的中间状态。

# 只有控制面能写该表；数据面只读。授权也绑定到同一个发布条目。
trusted_registry = {  # 计算并保存当前步骤的中间状态。
    artifact["artifact_id"]: {  # 执行当前语句以推进本节示例。
        "bundle_sha256": artifact["bundle_sha256"],  # 执行当前语句以推进本节示例。
        "model_version": artifact["versions"]["model"],  # 执行当前语句以推进本节示例。
        "config_sha256": artifact["config_sha256"],  # 执行当前语句以推进本节示例。
        "snapshot_sha256": artifact["training_snapshot"]["snapshot_sha256"],  # 执行当前语句以推进本节示例。
        "state_dict_sha256": artifact["state_dict_sha256"],  # 执行当前语句以推进本节示例。
        "series_by_tenant": {"tenant-a": ["gauge-17"]},  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。


def validate_artifact(candidate, candidate_model, registry, observed_snapshot=None):  # 定义本节可复用的核心函数。
    if not isinstance(candidate, dict):  # 按当前条件选择后续控制路径。
        raise RuntimeError("artifact_must_be_mapping")  # 遇到非法合同立即显式失败。
    artifact_id = candidate.get("artifact_id")  # 计算并保存当前步骤的中间状态。
    trusted = registry.get(artifact_id)  # 计算并保存当前步骤的中间状态。
    if trusted is None:  # 按当前条件选择后续控制路径。
        raise PermissionError("artifact_not_in_trusted_registry")  # 遇到非法合同立即显式失败。

    bundle_hash = candidate.get("bundle_sha256")  # 计算并保存当前步骤的中间状态。
    bundle_body = {key: value for key, value in candidate.items() if key != "bundle_sha256"}  # 计算并保存当前步骤的中间状态。
    if canonical_hash(bundle_body) != bundle_hash:  # 按当前条件选择后续控制路径。
        raise RuntimeError("artifact_bundle_hash_mismatch")  # 遇到非法合同立即显式失败。
    if bundle_hash != trusted["bundle_sha256"]:  # 按当前条件选择后续控制路径。
        raise PermissionError("artifact_bundle_not_trusted")  # 遇到非法合同立即显式失败。

    configuration = candidate.get("configuration")  # 计算并保存当前步骤的中间状态。
    if canonical_hash(configuration) != candidate.get("config_sha256"):  # 按当前条件选择后续控制路径。
        raise RuntimeError("configuration_hash_mismatch")  # 遇到非法合同立即显式失败。
    if candidate["config_sha256"] != trusted["config_sha256"]:  # 按当前条件选择后续控制路径。
        raise PermissionError("configuration_not_trusted")  # 遇到非法合同立即显式失败。
    if model_config(candidate_model) != configuration["model"]:  # 按当前条件选择后续控制路径。
        raise RuntimeError("model_configuration_mismatch")  # 遇到非法合同立即显式失败。

    actual_state_hash = state_dict_hash(candidate_model)  # 计算并保存当前步骤的中间状态。
    if actual_state_hash != candidate.get("state_dict_sha256"):  # 按当前条件选择后续控制路径。
        raise RuntimeError("model_state_dict_mismatch")  # 遇到非法合同立即显式失败。
    if actual_state_hash != trusted["state_dict_sha256"]:  # 按当前条件选择后续控制路径。
        raise PermissionError("model_state_not_trusted")  # 遇到非法合同立即显式失败。

    snapshot = candidate.get("training_snapshot")  # 计算并保存当前步骤的中间状态。
    snapshot_body = {key: value for key, value in snapshot.items() if key != "snapshot_sha256"}  # 计算并保存当前步骤的中间状态。
    if canonical_hash(snapshot_body) != snapshot.get("snapshot_sha256"):  # 按当前条件选择后续控制路径。
        raise RuntimeError("training_snapshot_hash_mismatch")  # 遇到非法合同立即显式失败。
    if snapshot["snapshot_sha256"] != trusted["snapshot_sha256"]:  # 按当前条件选择后续控制路径。
        raise PermissionError("training_snapshot_not_trusted")  # 遇到非法合同立即显式失败。
    if observed_snapshot is not None and observed_snapshot != snapshot:  # 按当前条件选择后续控制路径。
        raise RuntimeError("observed_training_data_or_features_mismatch")  # 遇到非法合同立即显式失败。

    if candidate["versions"]["model"] != trusted["model_version"]:  # 按当前条件选择后续控制路径。
        raise PermissionError("model_version_not_trusted")  # 遇到非法合同立即显式失败。
    return True  # 返回当前分支计算出的结果。


assert validate_artifact(artifact, model, trusted_registry, training_snapshot)  # 用受控断言验证关键不变量。
assert artifact["configuration"]["model"]["receptive_field"] <= WINDOW  # 用受控断言验证关键不变量。
assert len({artifact["bundle_sha256"], artifact["config_sha256"], artifact["state_dict_sha256"]}) == 3  # 用受控断言验证关键不变量。

# 参数被改、feature/data 快照被改、或攻击者重算出一个自洽但未发布的 bundle，都必须拒绝。
tampered_model = deepcopy(model)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    next(tampered_model.parameters()).add_(0.01)  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    validate_artifact(artifact, tampered_model, trusted_registry)  # 执行当前语句以推进本节示例。
    raise AssertionError("tampered parameters must fail closed")  # 遇到非法合同立即显式失败。
except RuntimeError as exc:  # 捕获预期异常并验证失败分支。
    assert str(exc) == "model_state_dict_mismatch"  # 用受控断言验证关键不变量。

tampered_features = all_features.copy()  # 计算并保存当前步骤的中间状态。
tampered_features[0, 0] += 1.0  # 计算并保存当前步骤的中间状态。
tampered_snapshot = make_training_snapshot(  # 计算并保存当前步骤的中间状态。
    raw_value, tampered_features, timestamps, target_indices,  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    validate_artifact(artifact, model, trusted_registry, tampered_snapshot)  # 执行当前语句以推进本节示例。
    raise AssertionError("tampered data/features must fail closed")  # 遇到非法合同立即显式失败。
except RuntimeError as exc:  # 捕获预期异常并验证失败分支。
    assert str(exc) == "observed_training_data_or_features_mismatch"  # 用受控断言验证关键不变量。

forged_model = TCNForecaster(3).to(DEVICE)  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    validate_artifact(artifact, forged_model, trusted_registry)  # 执行当前语句以推进本节示例。
    raise AssertionError("forged model must fail closed")  # 遇到非法合同立即显式失败。
except RuntimeError as exc:  # 捕获预期异常并验证失败分支。
    assert str(exc) == "model_state_dict_mismatch"  # 用受控断言验证关键不变量。

forged_artifact = deepcopy(artifact)  # 计算并保存当前步骤的中间状态。
forged_artifact["artifact_id"] = "attacker-self-signed-bundle"  # 计算并保存当前步骤的中间状态。
forged_body = {key: value for key, value in forged_artifact.items() if key != "bundle_sha256"}  # 计算并保存当前步骤的中间状态。
forged_artifact["bundle_sha256"] = canonical_hash(forged_body)  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    validate_artifact(forged_artifact, model, trusted_registry)  # 执行当前语句以推进本节示例。
    raise AssertionError("unregistered self-consistent bundle must fail closed")  # 遇到非法合同立即显式失败。
except PermissionError as exc:  # 捕获预期异常并验证失败分支。
    assert str(exc) == "artifact_not_in_trusted_registry"  # 用受控断言验证关键不变量。

## 11. 受信制品下的严格流式一步预测

窗口状态同时绑定 tenant、series_id 和已发布 artifact。新事件必须是有限数、带时区且恰好比 watermark 晚一个制品规定的采样间隔；缺口、乱序和跨主体访问立即拒绝。相同 event_id 与相同规范化 payload 返回 duplicate 且不会再次入窗，相同 ID 不同 payload 返回冲突。窗口长度只能来自 artifact，不能由调用方另传。ready 后按同一份 normalizer 和 UTC 小时公式构造 value_z/hour_sin/hour_cos，再在每次都通过完整性校验的模型上执行真实 one-step 推理。

In [ ]:
@dataclass  # 为下方定义附加声明式配置。
class StreamWindow:  # 定义承载本节状态与行为的数据结构。
    tenant: str  # 执行当前语句以推进本节示例。
    series_id: str  # 执行当前语句以推进本节示例。
    artifact: dict  # 执行当前语句以推进本节示例。
    model: nn.Module  # 执行当前语句以推进本节示例。
    registry: dict  # 执行当前语句以推进本节示例。
    records: list = field(default_factory=list)  # 计算并保存当前步骤的中间状态。
    seen: dict = field(default_factory=dict)  # 计算并保存当前步骤的中间状态。
    watermark: datetime | None = None  # 计算并保存当前步骤的中间状态。

    def __post_init__(self):  # 定义本节可复用的核心函数。
        validate_artifact(self.artifact, self.model, self.registry)  # 执行当前语句以推进本节示例。
        trusted = self.registry[self.artifact["artifact_id"]]  # 计算并保存当前步骤的中间状态。
        allowed_series = trusted["series_by_tenant"].get(self.tenant, [])  # 计算并保存当前步骤的中间状态。
        if self.series_id not in allowed_series:  # 按当前条件选择后续控制路径。
            raise PermissionError("tenant_series_not_authorized")  # 遇到非法合同立即显式失败。
        sampling = self.artifact["configuration"]["sampling"]  # 计算并保存当前步骤的中间状态。
        self.window = int(sampling["window"])  # 计算并保存当前步骤的中间状态。
        self.frequency = timedelta(seconds=int(sampling["frequency_seconds"]))  # 计算并保存当前步骤的中间状态。
        if self.window < 1 or self.frequency.total_seconds() <= 0:  # 按当前条件选择后续控制路径。
            raise RuntimeError("invalid_artifact_window_or_frequency")  # 遇到非法合同立即显式失败。

    @property  # 为下方定义附加声明式配置。
    def values(self):  # 定义本节可复用的核心函数。
        return [value for _, value in self.records]  # 返回当前分支计算出的结果。

    def _validate_event_identity(self, event):  # 定义本节可复用的核心函数。
        if event.get("tenant") != self.tenant:  # 按当前条件选择后续控制路径。
            raise PermissionError("tenant_mismatch")  # 遇到非法合同立即显式失败。
        if event.get("series_id") != self.series_id:  # 按当前条件选择后续控制路径。
            raise PermissionError("series_mismatch")  # 遇到非法合同立即显式失败。
        event_id = event.get("event_id")  # 计算并保存当前步骤的中间状态。
        if not isinstance(event_id, str) or not event_id:  # 按当前条件选择后续控制路径。
            raise ValueError("nonempty_event_id_required")  # 遇到非法合同立即显式失败。
        return event_id  # 返回当前分支计算出的结果。

    def append(self, event):  # 定义本节可复用的核心函数。
        # 每次写入前复验，构造窗口后发生的模型或 manifest 篡改也不能继续服务。
        validate_artifact(self.artifact, self.model, self.registry)  # 执行当前语句以推进本节示例。
        event_id = self._validate_event_identity(event)  # 计算并保存当前步骤的中间状态。
        timestamp = event.get("event_time")  # 计算并保存当前步骤的中间状态。
        if not isinstance(timestamp, datetime) or timestamp.tzinfo is None or timestamp.utcoffset() is None:  # 按当前条件选择后续控制路径。
            raise ValueError("timezone_required")  # 遇到非法合同立即显式失败。
        timestamp = timestamp.astimezone(timezone.utc)  # 计算并保存当前步骤的中间状态。
        value = float(event.get("value"))  # 计算并保存当前步骤的中间状态。
        if not math.isfinite(value):  # 按当前条件选择后续控制路径。
            raise ValueError("finite_value_required")  # 遇到非法合同立即显式失败。

        fingerprint = canonical_hash({  # 计算并保存当前步骤的中间状态。
            "tenant": self.tenant,  # 执行当前语句以推进本节示例。
            "series_id": self.series_id,  # 执行当前语句以推进本节示例。
            "event_time_utc": timestamp.isoformat(),  # 执行当前语句以推进本节示例。
            "value": value,  # 执行当前语句以推进本节示例。
        })  # 执行当前语句以推进本节示例。
        if event_id in self.seen:  # 按当前条件选择后续控制路径。
            if self.seen[event_id] != fingerprint:  # 按当前条件选择后续控制路径。
                raise ValueError("idempotency_conflict")  # 遇到非法合同立即显式失败。
            return "duplicate"  # 返回当前分支计算出的结果。

        if self.watermark is not None:  # 按当前条件选择后续控制路径。
            if timestamp <= self.watermark:  # 按当前条件选择后续控制路径。
                raise ValueError("out_of_order_requires_replay")  # 遇到非法合同立即显式失败。
            if timestamp != self.watermark + self.frequency:  # 按当前条件选择后续控制路径。
                raise ValueError("time_gap_requires_fill_or_replay")  # 遇到非法合同立即显式失败。

        self.seen[event_id] = fingerprint  # 计算并保存当前步骤的中间状态。
        self.records.append((timestamp, value))  # 执行当前语句以推进本节示例。
        self.records = self.records[-self.window:]  # 计算并保存当前步骤的中间状态。
        self.watermark = timestamp  # 计算并保存当前步骤的中间状态。
        return "ready" if len(self.records) == self.window else "warmup"  # 返回当前分支计算出的结果。

    def build_feature_matrix(self):  # 定义本节可复用的核心函数。
        validate_artifact(self.artifact, self.model, self.registry)  # 执行当前语句以推进本节示例。
        if len(self.records) != self.window:  # 按当前条件选择后续控制路径。
            raise RuntimeError("window_not_ready")  # 遇到非法合同立即显式失败。
        feature_spec = self.artifact["configuration"]["features"]  # 计算并保存当前步骤的中间状态。
        if feature_spec["order"] != ["value_z", "hour_sin", "hour_cos"]:  # 按当前条件选择后续控制路径。
            raise RuntimeError("unsupported_feature_contract")  # 遇到非法合同立即显式失败。
        mean = float(self.artifact["normalizer"]["mean"])  # 计算并保存当前步骤的中间状态。
        std = float(self.artifact["normalizer"]["std"])  # 计算并保存当前步骤的中间状态。
        if not math.isfinite(mean) or not math.isfinite(std) or std <= 0:  # 按当前条件选择后续控制路径。
            raise RuntimeError("invalid_artifact_normalizer")  # 遇到非法合同立即显式失败。

        rows = []  # 计算并保存当前步骤的中间状态。
        for timestamp, value in self.records:  # 遍历输入元素以累积或检查结果。
            seconds_in_day = (  # 计算并保存当前步骤的中间状态。
                timestamp.hour * 3600 + timestamp.minute * 60  # 执行当前语句以推进本节示例。
                + timestamp.second + timestamp.microsecond / 1_000_000  # 执行当前语句以推进本节示例。
            )  # 执行当前语句以推进本节示例。
            hour_utc = seconds_in_day / 3600  # 计算并保存当前步骤的中间状态。
            rows.append([  # 执行当前语句以推进本节示例。
                (value - mean) / std,  # 执行当前语句以推进本节示例。
                math.sin(2 * math.pi * hour_utc / feature_spec["hour_period"]),  # 执行当前语句以推进本节示例。
                math.cos(2 * math.pi * hour_utc / feature_spec["hour_period"]),  # 执行当前语句以推进本节示例。
            ])  # 执行当前语句以推进本节示例。
        matrix = np.asarray(rows, dtype=np.float32).T  # 计算并保存当前步骤的中间状态。
        if matrix.shape != (len(feature_spec["order"]), self.window):  # 按当前条件选择后续控制路径。
            raise RuntimeError("online_feature_shape_mismatch")  # 遇到非法合同立即显式失败。
        if not np.isfinite(matrix).all():  # 按当前条件选择后续控制路径。
            raise RuntimeError("nonfinite_online_features")  # 遇到非法合同立即显式失败。
        return matrix  # 返回当前分支计算出的结果。

    def predict_next(self):  # 定义本节可复用的核心函数。
        validate_artifact(self.artifact, self.model, self.registry)  # 执行当前语句以推进本节示例。
        matrix = self.build_feature_matrix()  # 计算并保存当前步骤的中间状态。
        was_training = self.model.training  # 计算并保存当前步骤的中间状态。
        self.model.eval()  # 执行当前语句以推进本节示例。
        with torch.no_grad():  # 在受管理的上下文中执行操作。
            prediction_z = float(self.model(torch.tensor(matrix[None], dtype=torch.float32))[0])  # 计算并保存当前步骤的中间状态。
        self.model.train(was_training)  # 执行当前语句以推进本节示例。
        mean = float(self.artifact["normalizer"]["mean"])  # 计算并保存当前步骤的中间状态。
        std = float(self.artifact["normalizer"]["std"])  # 计算并保存当前步骤的中间状态。
        return {  # 返回当前分支计算出的结果。
            "prediction_z": prediction_z,  # 执行当前语句以推进本节示例。
            "prediction": prediction_z * std + mean,  # 执行当前语句以推进本节示例。
            "target_time_utc": (self.watermark + self.frequency).isoformat(),  # 执行当前语句以推进本节示例。
            "trace": {  # 执行当前语句以推进本节示例。
                "tenant": self.tenant,  # 执行当前语句以推进本节示例。
                "series_id": self.series_id,  # 执行当前语句以推进本节示例。
                "artifact_id": self.artifact["artifact_id"],  # 执行当前语句以推进本节示例。
                "bundle_sha256": self.artifact["bundle_sha256"],  # 执行当前语句以推进本节示例。
                "model_version": self.artifact["versions"]["model"],  # 执行当前语句以推进本节示例。
            },  # 执行当前语句以推进本节示例。
        }  # 执行当前语句以推进本节示例。


buffer = StreamWindow(  # 计算并保存当前步骤的中间状态。
    tenant="tenant-a", series_id="gauge-17",  # 计算并保存当前步骤的中间状态。
    artifact=artifact, model=model, registry=trusted_registry,  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
events_live = [  # 计算并保存当前步骤的中间状态。
    {  # 执行当前语句以推进本节示例。
        "event_id": f"live-{i}",  # 执行当前语句以推进本节示例。
        "tenant": "tenant-a",  # 执行当前语句以推进本节示例。
        "series_id": "gauge-17",  # 执行当前语句以推进本节示例。
        "event_time": timestamps[i],  # 执行当前语句以推进本节示例。
        "value": float(raw_value[i]),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    for i in range(WINDOW)  # 遍历输入元素以累积或检查结果。
]  # 执行当前语句以推进本节示例。
statuses = [buffer.append(event) for event in events_live]  # 计算并保存当前步骤的中间状态。
assert statuses[:-1] == ["warmup"] * (WINDOW - 1) and statuses[-1] == "ready"  # 用受控断言验证关键不变量。
assert buffer.window == artifact["configuration"]["sampling"]["window"] == WINDOW  # 用受控断言验证关键不变量。
assert buffer.append(events_live[-1]) == "duplicate"  # 用受控断言验证关键不变量。
assert len(buffer.records) == WINDOW  # 用受控断言验证关键不变量。

# 在线 feature 与第一个离线监督窗口逐元素一致，随后真正执行一步预测。
online_matrix = buffer.build_feature_matrix()  # 计算并保存当前步骤的中间状态。
np.testing.assert_allclose(online_matrix, all_features[:WINDOW].T, rtol=0, atol=1e-6)  # 计算并保存当前步骤的中间状态。
stream_result = buffer.predict_next()  # 计算并保存当前步骤的中间状态。
model.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    offline_prediction_z = float(model(X_all[:1])[0])  # 计算并保存当前步骤的中间状态。
assert abs(stream_result["prediction_z"] - offline_prediction_z) < 1e-6  # 用受控断言验证关键不变量。
assert stream_result["target_time_utc"] == timestamps[WINDOW].isoformat()  # 用受控断言验证关键不变量。
assert math.isfinite(stream_result["prediction"])  # 用受控断言验证关键不变量。

state_before_rejections = (list(buffer.records), dict(buffer.seen), buffer.watermark)  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    buffer.append({**events_live[-1], "value": 999.0})  # 执行当前语句以推进本节示例。
    raise AssertionError("same event id with different payload must fail")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert str(exc) == "idempotency_conflict"  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    buffer.append({**events_live[-1], "event_id": "wrong-tenant", "tenant": "tenant-b"})  # 执行当前语句以推进本节示例。
    raise AssertionError("cross tenant must fail")  # 遇到非法合同立即显式失败。
except PermissionError as exc:  # 捕获预期异常并验证失败分支。
    assert str(exc) == "tenant_mismatch"  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    buffer.append({**events_live[-1], "event_id": "wrong-series", "series_id": "gauge-18"})  # 执行当前语句以推进本节示例。
    raise AssertionError("cross series must fail")  # 遇到非法合同立即显式失败。
except PermissionError as exc:  # 捕获预期异常并验证失败分支。
    assert str(exc) == "series_mismatch"  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    buffer.append({  # 执行当前语句以推进本节示例。
        **events_live[-1], "event_id": "nan-value",  # 执行当前语句以推进本节示例。
        "event_time": buffer.watermark + buffer.frequency, "value": float("nan"),  # 执行当前语句以推进本节示例。
    })  # 执行当前语句以推进本节示例。
    raise AssertionError("NaN must fail")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert str(exc) == "finite_value_required"  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    buffer.append({  # 执行当前语句以推进本节示例。
        **events_live[-1], "event_id": "inf-value",  # 执行当前语句以推进本节示例。
        "event_time": buffer.watermark + buffer.frequency, "value": float("inf"),  # 执行当前语句以推进本节示例。
    })  # 执行当前语句以推进本节示例。
    raise AssertionError("Inf must fail")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert str(exc) == "finite_value_required"  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    buffer.append({  # 执行当前语句以推进本节示例。
        **events_live[-1], "event_id": "time-gap",  # 执行当前语句以推进本节示例。
        "event_time": buffer.watermark + 2 * buffer.frequency,  # 执行当前语句以推进本节示例。
    })  # 执行当前语句以推进本节示例。
    raise AssertionError("time gap must fail")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert str(exc) == "time_gap_requires_fill_or_replay"  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    buffer.append({  # 执行当前语句以推进本节示例。
        **events_live[-1], "event_id": "out-of-order",  # 执行当前语句以推进本节示例。
        "event_time": buffer.watermark,  # 执行当前语句以推进本节示例。
    })  # 执行当前语句以推进本节示例。
    raise AssertionError("out of order event must fail")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert str(exc) == "out_of_order_requires_replay"  # 用受控断言验证关键不变量。
assert (buffer.records, buffer.seen, buffer.watermark) == state_before_rejections  # 用受控断言验证关键不变量。

## 12. 来源与生产边界

- Bai, Kolter 与 Koltun，An Empirical Evaluation of Generic Convolutional and Recurrent Networks for Sequence Modeling：https://arxiv.org/abs/1803.01271
- van den Oord 等，WaveNet：https://arxiv.org/abs/1609.03499
- PyTorch Conv1d 官方文档：https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html
- PyTorch reproducibility：https://pytorch.org/docs/stable/notes/randomness.html

本例没有多变量业务事件、概率预测、多 horizon、节假日、缺失训练或滚动回测；test 数字只回答“上一真实观测已经到达时的下一点”问题，不代表 recursive 多步效果。内存 registry 演示的是验证顺序和 fail-closed 合同，不等于生产密钥基础设施；真实系统还要使用签名制品、不可变对象存储、受控发布身份、状态持久化、并发锁和 replay/correction 管道。生产替换模型后仍应保留因果干预测试、时间切分、离线/在线 feature parity 与制品校验。

In [ ]:
assert X_train.shape[2] == WINDOW  # 用受控断言验证关键不变量。
assert target_indices[train_mask].max() < target_indices[valid_mask].min()  # 用受控断言验证关键不变量。
assert target_indices[valid_mask].max() < target_indices[test_mask].min()  # 用受控断言验证关键不变量。
assert model(X_train[:4]).shape == (4,)  # 用受控断言验证关键不变量。
assert first_grad_norm > 0  # 用受控断言验证关键不变量。
assert train_losses[-1] < train_losses[0]  # 用受控断言验证关键不变量。
assert math.isfinite(best_validation)  # 用受控断言验证关键不变量。
assert tcn_metrics["mae"] < 1.5  # 用受控断言验证关键不变量。

# 第二个 test 窗口使用第一个 test target 的真实到达值：这是 rolling one-step，不是递归多步。
assert test_target_indices[0] == VALID_END  # 用受控断言验证关键不变量。
assert torch.isclose(X_test[1, 0, -1], torch.tensor(normalized_value[VALID_END]))  # 用受控断言验证关键不变量。
assert validate_artifact(artifact, model, trusted_registry, training_snapshot)  # 用受控断言验证关键不变量。
assert artifact["configuration"]["features"]["order"][0] == "value_z"  # 用受控断言验证关键不变量。
assert buffer.watermark == events_live[-1]["event_time"]  # 用受控断言验证关键不变量。
assert buffer.values == [float(value) for value in raw_value[:WINDOW]]  # 用受控断言验证关键不变量。
assert stream_result["trace"]["tenant"] == "tenant-a"  # 用受控断言验证关键不变量。
assert stream_result["trace"]["series_id"] == "gauge-17"  # 用受控断言验证关键不变量。

# 即使对象构造时可信，之后发生权重篡改，下一次实际推理仍必须 fail-closed。
post_init_tampered_model = deepcopy(model)  # 计算并保存当前步骤的中间状态。
guarded_buffer = StreamWindow(  # 计算并保存当前步骤的中间状态。
    tenant="tenant-a", series_id="gauge-17",  # 计算并保存当前步骤的中间状态。
    artifact=artifact, model=post_init_tampered_model, registry=trusted_registry,  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    next(post_init_tampered_model.parameters()).mul_(0.0)  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    guarded_buffer.predict_next()  # 执行当前语句以推进本节示例。
    raise AssertionError("post-init parameter tamper must fail before prediction")  # 遇到非法合同立即显式失败。
except RuntimeError as exc:  # 捕获预期异常并验证失败分支。
    assert str(exc) == "model_state_dict_mismatch"  # 用受控断言验证关键不变量。

print({  # 执行当前语句以推进本节示例。
    "status": "all_checks_passed",  # 执行当前语句以推进本节示例。
    "evaluation": "rolling-one-step-teacher-forcing",  # 执行当前语句以推进本节示例。
    "artifact_id": artifact["artifact_id"],  # 执行当前语句以推进本节示例。
    "stream_prediction": round(stream_result["prediction"], 4),  # 执行当前语句以推进本节示例。
})  # 执行当前语句以推进本节示例。